In [ ]:
import numpy as np
from scipy.special import i0, expit
from scipy.stats import vonmises
import pandas as pd

pheno = pd.read_parquet("phenocam_phenology_model.parquet")
climate = pd.read_parquet("phenocam_site_in_usa_climate.parquet")
climate.columns = ["year", "yday", "prcp", "tmax", "tmin", "vp", "site"]

### Prepare data

In [ ]:
# Reshape long-format climate data to one row per site-year,
# with array columns containing the full daily time series.
climate_arrays = (
    climate
    .sort_values(["site", "year", "yday"])
    .groupby(["site", "year"], as_index=False)
    .apply(lambda g: pd.Series({
        "tmax": g["tmax"].values,
        "prcp": g["prcp"].values
    }))
)

print(climate_arrays.shape)
climate_arrays.head()

In [ ]:
# Select only the phenology columns we need and join to climate_arrays.
pheno_subset = pheno[["site", "transition_year_rising", "veg_type", "doy_transition_50_rising"]].copy()

model_data = climate_arrays.merge(
    pheno_subset,
    left_on=["site", "year"],
    right_on=["site", "transition_year_rising"],
    how="inner"
)

# Drop the redundant year column from the phenology table
model_data = model_data.drop(columns=["transition_year_rising"])

print(model_data.shape)
model_data.head()

### Model definition

In [ ]:
DOY_MAX = 365  # could also use array length per row

def greenup_activity(temperature: np.ndarray[float], precipitation: np.ndarray[float], 
                         tmin: float, 
                         temp_sigmoid_inflection: float, temp_sigmoid_slope: float,
                         prcp_sigmoid_inflection: float, prcp_sigmoid_slope: float,
                         threshold: float,
                         verbose: bool=False):
    """Predict greenup date from cumulative temperature and precipitation forcing.

    Temperature forcing accumulates as degree-days above `tmin`. All precipitation
    contributes directly to forcing. Greenup is predicted as the first day when
    cumulative forcing reaches `threshold`.

    Parameters
    ----------
    temperature : array-like
        Daily air temperature time series.
    precipitation : array-like
        Daily precipitation time series, same length as `temperature`.
    tmin : float
        Minimum temperature threshold for degree-day accumulation (fitted parameter).
    threshold : float
        Critical cumulative forcing threshold that triggers greenup (fitted parameter).

    Returns
    -------
    int or float
        Index of the first day when cumulative forcing >= `threshold`.
        Returns `np.nan` if the threshold is never reached.
    """
    #temperature = np.asarray(temperature, dtype=float)
    #precipitation = np.asarray(precipitation, dtype=float)

    if temperature.shape != precipitation.shape:
        raise ValueError("temperature and precipitation must have the same shape")

    # Degree-day temperature forcing, clipped at zero
    temp_forcing = np.maximum(temperature - tmin, 0.0)
    temp_forcing_cumulative = np.cumsum(temp_forcing, axis=1)

    # Precipitation forcing is just the sum
    prcp_forcing_cumulative = np.cumsum(precipitation, axis=1)

    # Calculate sigomid responses
    temp_forcing_sigmoid = expit(temp_sigmoid_slope * (temp_forcing_cumulative - temp_sigmoid_inflection))
    prcp_forcing_sigmoid = expit(prcp_sigmoid_slope * (prcp_forcing_cumulative - prcp_sigmoid_inflection))

    # Multiply
    total_forcing = temp_forcing_sigmoid * prcp_forcing_sigmoid

    if verbose:
        return temp_forcing_sigmoid, prcp_forcing_sigmoid, total_forcing
    else:
        return total_forcing

In [ ]:
# Synthetic demonstration of predict_greenup_date
temp = np.array([2, 5, 8, 12, 15, 10, 18, 20, 22, 19]).reshape(2, -1)
precip = np.array([0, 1, 3, 2, 5, 5, 1, 4, 3, 3]).reshape(2, -1)

greenup_idx = greenup_activity(temp, precip, tmin=5.0, temp_sigmoid_inflection=10, temp_sigmoid_slope=0.5, prcp_sigmoid_inflection=8, prcp_sigmoid_slope=0.5, threshold=0.5)

print(f"Predicted greenup day index: {greenup_idx}")
print(f"Temperature: {temp}")
print(f"Precipitation: {precip}")

In [ ]:
greenup_idx.shape

In [ ]:
np.take_along_axis(greenup_idx, np.array([[0], [2]]), axis=1)

In [ ]:
from scipy.optimize import minimize

def doy_to_radians(doy, doy_max=DOY_MAX):
    """Convert day-of-year to radians on [0, 2*pi)."""
    return 2 * np.pi * (doy - 1) / doy_max

def nll(params, observed_doy, tmax_arrays, prcp_arrays, kappa, doy_max=DOY_MAX):
    """Negative log-likelihood for Von Mises response."""
    tmin, temp_inflection, temp_slope, prcp_inflection, prcp_slope, threshold = params
    
    if kappa <= 0 or threshold <= 0:
        return np.inf
    
    activity_ts = greenup_activity(tmax_arrays, prcp_arrays, *params)
    activity_at_greenup = np.take_along_axis(activity_ts, (observed_doy-1).reshape(-1, 1), 1)

    error = np.sum((activity_at_greenup - 0.5)**2)
    return error

    '''
    # Drop rows where prediction failed
    valid = ~np.isnan(predicted_doy)
    if not valid.any():
        return np.inf
    
    obs_rad = doy_to_radians(observed_doy[valid], doy_max)
    pred_rad = doy_to_radians(predicted_doy[valid], doy_max)
    
    # Circular difference: ensure obs_rad and pred_rad are on same circle
    # scipy vonmises handles loc parameter, so we can use loc=pred_rad
    logpdf = vonmises.logpdf(obs_rad, kappa, loc=pred_rad)
    return -np.sum(logpdf)
    '''

In [ ]:
tmax_arrays = np.stack(model_data["tmax"])
prcp_arrays = np.stack(model_data["prcp"])

# Initial guess
x0 = [5.0, 100.0, 0.1, 50, 0.1, 0.5]  # tmin, temp_inflection/slope, prcp_inflection/slope, threshold

# Bounds
bounds = [(-10, 20), (10, 1000), (1e-3, 1.0), (10, 1000), (1e-3, 1.0), (0, 1.0)]

global_result = minimize(
    nll,
    x0,
    args=(
        model_data["doy_transition_50_rising"].values, 
        tmax_arrays, 
        prcp_arrays,
        5.0
    ),
    method="L-BFGS-B",
    bounds=bounds
)

print(global_result)

In [ ]:
global_yhat = np.argmax(greenup_activity(tmax_arrays, prcp_arrays, *global_result.x) >= 0.5, axis=1)+1

To fit by each veg type.

In [ ]:
results = []

for grp, subdf in model_data.groupby("veg_type"):
    if subdf.shape[0] > 100:
        model_subdf = subdf.copy()
    else:
        model_subdf = model_data.copy()
        
    tmax_arrays = np.stack(model_subdf["tmax"])
    prcp_arrays = np.stack(model_subdf["prcp"])
    
    # Keep initial guess and bounds from global model
    result = minimize(
        nll,
        x0,
        args=(model_subdf["doy_transition_50_rising"].values, 
              tmax_arrays, 
              prcp_arrays,
             5.0),
        method="L-BFGS-B",
        bounds=bounds
    )

    model_subdf["model_doy_transition_50_rising"] = np.argmax(greenup_activity(tmax_arrays, prcp_arrays, *result.x) > 0.5, axis=1) + 1
    
    results.append(model_subdf[model_subdf.veg_type == grp])

veg_type_results = pd.concat(results, axis=0)

In [ ]:
from matplotlib import pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, sharex=True, sharey=True)

ax1.scatter(model_data["doy_transition_50_rising"] % DOY_MAX, global_yhat, s=1)
ax2.scatter(veg_type_results["doy_transition_50_rising"] % DOY_MAX, veg_type_results["model_doy_transition_50_rising"], s=1)

ax1.set_title("Global")
ax2.set_title("By veg type")

for a in (ax1, ax2):
    a.set_xlim(0, 365)
    a.set_ylim(0, 365)
    a.set_aspect("equal")
    a.grid()
    a.axline((0, 0), (1, 1), color="black")
    
fig.supxlabel("Observed greenup DOY")
fig.supylabel("Model greenup DOY")

plt.tight_layout()
plt.show()

In [ ]:
def jammalamadaka_sharma_corr(alpha, beta, cycle_length=365):
    """
    Computes the Jammalamadaka-Sharma circular correlation coefficient.
    """
    alpha = np.asarray(alpha)
    beta = np.asarray(beta)

    alpha = alpha * (2 * np.pi) / cycle_length
    beta  = beta * (2 * np.pi) / cycle_length
    
    # Compute circular means using mean of sines and cosines
    mean_alpha = np.arctan2(np.mean(np.sin(alpha)), np.mean(np.cos(alpha)))
    mean_beta = np.arctan2(np.mean(np.sin(beta)), np.mean(np.cos(beta)))
    
    # Compute deviations
    sin_alpha_dev = np.sin(alpha - mean_alpha)
    sin_beta_dev = np.sin(beta - mean_beta)
    
    # Calculate numerator and denominator components
    num = np.sum(sin_alpha_dev * sin_beta_dev)
    den = np.sqrt(np.sum(sin_alpha_dev**2) * np.sum(sin_beta_dev**2))
    
    if den == 0:
        return 0.0
    return num / den

def angular_absolute_error(y_true, y_hat, cycle_length=365):
    to_rad = (2 * np.pi) / cycle_length
    
    y_true_rad = y_true * to_rad
    y_hat_rad  = y_hat  * to_rad

    error_rad = np.minimum(np.abs(y_true_rad - y_hat_rad), 2*np.pi - np.abs(y_true_rad - y_hat_rad))

    return error_rad / to_rad

In [ ]:
jammalamadaka_sharma_corr(model_data["doy_transition_50_rising"], global_yhat)

In [ ]:
angular_absolute_error(model_data["doy_transition_50_rising"], global_yhat).mean()

In [ ]:
jammalamadaka_sharma_corr(veg_type_results["doy_transition_50_rising"], veg_type_results["model_doy_transition_50_rising"])

In [ ]:
angular_absolute_error(veg_type_results["doy_transition_50_rising"], veg_type_results["model_doy_transition_50_rising"]).mean()

### Visualize forcing sigmoids

In [ ]:
temp_forcing, prcp_forcing, total_forcing = greenup_activity(tmax_arrays, prcp_arrays, *global_result.x, verbose=True)

In [ ]:
plt.plot(temp_forcing[:100, :].T, alpha=0.5, color="orange")
plt.plot(prcp_forcing[:100, :].T, alpha=0.5, color="blue")
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True)
x = np.arange(365)+1
ax1.plot(x, tmax_arrays[0, :], label="Tmax", color="orange")
ax_prcp = ax1.twinx()
ax_prcp.plot(x, prcp_arrays[0, :], label="Prcp", color="blue")

ax1.set_ylabel("Tmax (deg C)")
ax_prcp.set_ylabel("Prcp (mm)")

ax2.plot(temp_forcing[0, :], label="Tmax", color="orange")
ax2.plot(prcp_forcing[0, :], label="Prcp", color="blue")
ax2.plot(total_forcing[0, :], label="Total", color="black")
ax2.set_ylim(0, None)

ax2.legend()

fig.supxlabel("DOY")

ax2.set_ylabel("Relative forcing")

plt.show()